# Лекция 3. Разбор упражнений

Упражнения из раздела 12 конспекта [`lecture03.md`](lecture03.md). Каждое разбирается по одной схеме:

1. **условие**;
2. **решение по шагам** — вручную, как на лекции;
3. **проверка кодом** — тот же вывод получаем численно.

Сначала попробуйте решить сами. Краткие ответы без кода — в [`exercises03.md`](exercises03.md); как считать множители из баланса сил в коде — в [`demo03.ipynb`](demo03.ipynb).

In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize, minimize_scalar

BLUE, ORANGE, AQUA, RED, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#8a8985", "#0b0b0b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "legend.frameon": False})

## 12.1. Завод с контрактом: множитель как сила и как цена

> $\min_x (x-2)^2$ при $x\ge b$. Для $b=2.5$, $3$, $4$ найдите $x^\ast$ и $\mu^\ast$ из баланса сил $f'(x^\ast)=\mu^\ast h'(x^\ast)$. Убедитесь, что $\mu^\ast=df^\ast/db$. Что при $b<2$?

**Решение.** $h(x)=x-b$, $h'=1$. Безусловный минимум $x=2$. При $b>2$ он за стенкой, решение у стенки: $x^\ast=b$, и баланс сил даёт $\mu^\ast=f'(b)=2(b-2)$: для $b=2.5,3,4$ — $\mu^\ast=1,\,2,\,4$. Оптимальная себестоимость $f^\ast(b)=(b-2)^2$, её производная $2(b-2)$ — в точности $\mu^\ast$: множитель — цена ужесточения контракта на единицу. При $b<2$ безусловный минимум допустим, $x^\ast=2$, ограничение не активно ($h(x^\ast)=2-b>0$), и по комплементарной нежёсткости $\mu^\ast=0$; $f^\ast(b)=0$ не зависит от $b$ — цена ноль, что и логично: контракт не мешает.

In [2]:
f = lambda x: (x - 2) ** 2
fprime = lambda x: 2 * (x - 2)
print("   b    x*    mu* (сила)   df*/db (цена, конечная разность)")
for b in (1.0, 2.5, 3.0, 4.0):
    xstar = minimize_scalar(f, bounds=(b, b + 10), method="bounded", options={"xatol": 1e-10}).x
    mu = fprime(xstar) if np.isclose(xstar, b, atol=1e-6) else 0.0
    eps = 1e-6
    fstar = lambda bb: minimize_scalar(f, bounds=(bb, bb + 10), method="bounded", options={"xatol": 1e-10}).fun
    dfdb = (fstar(b + eps) - fstar(b - eps)) / (2 * eps)
    print(f"{b:5.1f}  {xstar:5.2f}  {mu:9.3f}   {dfdb:9.3f}")

   b    x*    mu* (сила)   df*/db (цена, конечная разность)
  1.0   2.00      0.000      -0.000
  2.5   2.50      1.000       1.000
  3.0   3.00      2.000       2.000
  4.0   4.00      4.000       4.000


## 12.2. LP планирования производства: где ломается теневая цена

> Увеличьте мощность третьего цеха с $18$ до $b_3$. Найдите значение $b_3$, при котором набор активных ограничений меняется, и новые множители сразу после этой точки. Почему множитель ограничения 3 обязана обратиться в ноль ровно там, где оно перестаёт быть активным?

**Решение.** Пока активны ограничения 2 ($2x_2\le12$) и 3 ($3x_1+2x_2\le18$), решение — их пересечение: $x_2=6$, $x_1=(b_3-12)/3$. Это движение по ребру продолжается, пока $x_1$ не упрётся в своё же ограничение $x_1\le4$, то есть пока $(b_3-12)/3\le4 \Leftrightarrow b_3\le24$. При $b_3=24$ решение — вершина $(4,6)$, где **одновременно** активны ограничения 1 и 2, а ограничение 3 перестаёт быть узким местом: дальнейшее увеличение $b_3$ ничего не меняет, оно больше не активно, и стенка, которой точка не касается, не давит: по комплементарной нежёсткости обязано быть $\mu_3^\ast=0$. Сразу после точки перелома ($b_3>24$) активны ограничения 1 и 2: решая систему $x_1=4$, $2x_2=12$, множители находятся из $A^\top y=c$ на активных строках: $y_1\cdot1+y_2\cdot0=3$, $y_1\cdot0+y_2\cdot2=5\Rightarrow y=(3,\,2.5,\,0)$.

In [3]:
A = np.array([[1.0, 0.0], [0.0, 2.0], [3.0, 2.0]])
c = np.array([3.0, 5.0])

for b3 in (20, 23, 24, 25, 28):
    r = linprog(-c, A_ub=A, b_ub=[4.0, 12.0, b3], bounds=[(0, None)] * 2, method="highs")
    y = -r.ineqlin.marginals
    print(f"b3={b3:5.1f}: x*={r.x.round(3)}, прибыль={-r.fun:.3f}, y*={y.round(3)}")

b3= 20.0: x*=[2.667 6.   ], прибыль=38.000, y*=[0.  1.5 1. ]
b3= 23.0: x*=[3.667 6.   ], прибыль=41.000, y*=[0.  1.5 1. ]
b3= 24.0: x*=[4. 6.], прибыль=42.000, y*=[3.  2.5 0. ]
b3= 25.0: x*=[4. 6.], прибыль=42.000, y*=[3.  2.5 0. ]
b3= 28.0: x*=[4. 6.], прибыль=42.000, y*=[3.  2.5 0. ]


## 12.3. QP на ящике: множители из линейной системы

> $Q=\begin{pmatrix}1&0.8\\0.8&2\end{pmatrix}$, $c=(1,-2)$, $-1\le x\le1$. Решите численно, определите активные грани и найдите множители из $\nabla f(x^\ast)=\sum_i\mu_i\nabla h_i(x^\ast)$ по активным $i$. Проверьте $\mu\ge0$ и комплементарную нежёсткость.

**Решение.** $Q\succ0$ (собственные числа $\approx0.557,\,2.443$), задача выпуклая. Численно $x^\ast=(-1,\,1)$ — вершина ящика: активны **две** грани, $x_1\ge-1$ ($h_2=1+x_1$, $\nabla h_2=(1,0)$) и $x_2\le1$ ($h_3=1-x_2$, $\nabla h_3=(0,-1)$). Градиент цели $\nabla f(x^\ast)=Qx^\ast+c=(-1+0.8+1,\ -0.8+2-2)=(0.8,\,-0.8)$. Баланс сил по двум активным нормалям: $(0.8,-0.8)=\mu_2(1,0)+\mu_3(0,-1)$, откуда $\mu_2=\mu_3=0.8>0$ — обе стенки давят внутрь, точка действительно оптимум. У двух неактивных граней $\mu_1=\mu_4=0$. Здесь конус нормалей двумерный, и $\nabla f$ лежит строго внутри него — типичная вершина.

In [4]:
Q = np.array([[1.0, 0.8], [0.8, 2.0]])
c = np.array([1.0, -2.0])
print("собственные числа Q:", np.linalg.eigvalsh(Q).round(4), "-> Q > 0")

fqp = lambda x: 0.5 * x @ Q @ x + c @ x
xstar = minimize(fqp, x0=np.zeros(2), jac=lambda x: Q @ x + c, method="L-BFGS-B", bounds=[(-1, 1), (-1, 1)]).x
h = np.array([1 - xstar[0], 1 + xstar[0], 1 - xstar[1], 1 + xstar[1]])
grad_h = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]], float)
active = np.isclose(h, 0, atol=1e-6)
print("x* =", xstar.round(4), " f* =", round(fqp(xstar), 6), " активные грани:", np.where(active)[0] + 1)

grad_f = Q @ xstar + c
mu = np.zeros(4)
mu[active] = np.linalg.solve(grad_h[active].T, grad_f)          # два уравнения, два неизвестных
print("grad f(x*) =", grad_f.round(4))
print("mu =", mu.round(4), " -> все >= 0")
print("остаток баланса:", (grad_f - mu @ grad_h).round(10))
print("mu_i * h_i(x*) =", (mu * h).round(8), "-> комплементарная нежёсткость выполнена")

собственные числа Q: [0.5566 2.4434] -> Q > 0
x* = [-1.  1.]  f* = -2.3  активные грани: [2 3]
grad f(x*) = [ 0.8 -0.8]
mu = [0.  0.8 0.8 0. ]  -> все >= 0
остаток баланса: [0. 0.]
mu_i * h_i(x*) = [0. 0. 0. 0.] -> комплементарная нежёсткость выполнена


## 12.4. Центр Чебышёва: баланс сил по $(c, r)$

> Переменные $(c,r)$, цель $-r$, ограничения $h_i=b_i-a_i^\top c-r\lVert a_i\rVert\ge0$. Выпишите баланс сил и получите $\sum_i\mu_ia_i=0$, $\sum_i\mu_i\lVert a_i\rVert=1$. Сверьте с `marginals`. Почему у двух сторон множители нулевые?

**Решение.** Градиент цели по $(c,r)$: $\nabla f=(0,0,-1)$. Градиенты ограничений: $\nabla h_i=(-a_i,\,-\lVert a_i\rVert)$. Баланс $\nabla f=\sum_i\mu_i\nabla h_i$ по координатам: по $c$ — $0=-\sum_i\mu_ia_i$, по $r$ — $-1=-\sum_i\mu_i\lVert a_i\rVert$. Отсюда обе формулы: нормали сторон, взвешенные множителями, уравновешивают друг друга (круг «зажат» с разных сторон), а сумма весов, нормированных длинами, равна единице. Стороны 4 и 5 круг не касается — их $h_i>0$, и по комплементарной нежёсткости $\mu_4=\mu_5=0$: стенка, которой не касаются, не давит. Единственность центра здесь следует из того, что три активные нормали не лежат в одной полуплоскости — круг «зажат».

In [5]:
A = np.array([[-1.0, 0.0], [0.0, -1.0], [1.0, 2.0], [3.0, 1.0], [1.0, -1.0]])
b = np.array([0.0, 0.0, 8.0, 12.0, 3.0])
norms = np.linalg.norm(A, axis=1)
res = linprog([0, 0, -1], A_ub=np.c_[A, norms], b_ub=b, bounds=[(None, None), (None, None), (0, None)], method="highs")
mu = -res.ineqlin.marginals
print("c* =", res.x[:2].round(4), " r* =", round(res.x[2], 4), " активны стороны:", np.where(res.slack < 1e-9)[0] + 1)
print("mu =", mu.round(4))
print("sum mu_i a_i     =", (mu @ A).round(8), " (должно быть 0)")
print("sum mu_i |a_i|   =", round(mu @ norms, 8), " (должно быть 1)")

c* = [1.5279 1.5279]  r* = 1.5279  активны стороны: [1 2 3]
mu = [0.191 0.382 0.191 0.    0.   ]
sum mu_i a_i     = [0. 0.]  (должно быть 0)
sum mu_i |a_i|   = 1.0  (должно быть 1)


## 12.5. Прямоугольник в круге: касание и знак множителя

> $\min -4x_1x_2$ при $x_1^2+x_2^2=1$, $x_1,x_2\ge0$. Проверьте систему раздела 6 в точках $(1/\sqrt2,1/\sqrt2)$, $(1,0)$, $(0,1)$; найдите $\lambda$ и $\mu$; где система выполнена, а где нарушен знак $\mu$?

**Решение.** $\nabla f=(-4x_2,\,-4x_1)$, $\nabla g=(2x_1,\,2x_2)$.

- В $(1/\sqrt2,1/\sqrt2)$ неравенства неактивны ($\mu=0$), остаётся $\nabla f=\lambda\nabla g$: $(-2\sqrt2,-2\sqrt2)=\lambda(\sqrt2,\sqrt2)$, $\lambda=-2$. Система выполнена — это минимум, $f=-2$ (квадрат — прямоугольник наибольшей площади).
- В $(1,0)$ активно ещё $x_2\ge0$ с нормалью $(0,1)$: $(0,-4)=\lambda(2,0)+\mu(0,1)$ даёт $\lambda=0$, $\mu=-4<0$. Стенка должна была бы *притягивать* — система нарушена. И правда, $f=0$ здесь **максимум** на дуге.
- В $(0,1)$ симметрично: $\mu=-4$ у стенки $x_1\ge0$, тоже максимум.

Итого единственная точка, где система выполнена целиком (включая знаки), — квадрат. Знак множителя отсеивает максимумы; седла и локальные минимумы без ограничений на знак различают условия второго порядка (лекция 8). В ДЗ 2 та же задача решалась через выпуклость: там мы выяснили, что задача невыпукла (нелинейное равенство) — и тем не менее система ККТ здесь находит ответ; это необходимое условие, оно работает и для невыпуклых задач.

In [6]:
f_rect = lambda x: -4 * x[0] * x[1]
grad_f = lambda x: np.array([-4 * x[1], -4 * x[0]])
grad_g = lambda x: 2 * x
res = minimize(f_rect, x0=[0.9, 0.1], bounds=[(0, 1), (0, 1)],
               constraints=[{"type": "eq", "fun": lambda x: x[0] ** 2 + x[1] ** 2 - 1}])
print("численно: x* =", res.x.round(4), " f* =", round(res.fun, 4))

for p, active_wall in (((1 / np.sqrt(2), 1 / np.sqrt(2)), None), ((1.0, 0.0), np.array([0.0, 1.0])), ((0.0, 1.0), np.array([1.0, 0.0]))):
    p = np.array(p)
    if active_wall is None:
        lam = (grad_f(p) @ grad_g(p)) / (grad_g(p) @ grad_g(p))
        ok = np.allclose(grad_f(p), lam * grad_g(p))
        print(f"{p.round(3)}: lambda = {lam:.2f}, неравенства неактивны -> система {'выполнена' if ok else 'нарушена'}, f = {f_rect(p):.2f}")
    else:
        lam, mu = np.linalg.solve(np.c_[grad_g(p), active_wall], grad_f(p))
        print(f"{p.round(3)}: lambda = {lam:.2f}, mu = {mu:.2f} {'>= 0' if mu >= 0 else '< 0 -> знак нарушен, не минимум'}, f = {f_rect(p):.2f}")

численно: x* = [0.7071 0.7071]  f* = -2.0
[0.707 0.707]: lambda = -2.00, неравенства неактивны -> система выполнена, f = -2.00
[1. 0.]: lambda = 0.00, mu = -4.00 < 0 -> знак нарушен, не минимум, f = -0.00
[0. 1.]: lambda = 0.00, mu = -4.00 < 0 -> знак нарушен, не минимум, f = -0.00


## Что запомнить

- Сначала — где живёт минимум: сравните безусловный минимум с допустимым множеством; если он снаружи, решение на границе, и часть ограничений активна (12.1 при $b>2$ против $b<2$).
- Множители считаются из **линейной системы по активным ограничениям**: $\nabla f(x^\ast)=\sum_{i\ \text{акт.}}\mu_i\nabla h_i(x^\ast)$ (12.3, 12.4); у неактивных $\mu_i=0$ обязательно.
- Знак $\mu\ge0$ — содержательное условие: он отсеивает максимумы (12.5) и меняется вместе с набором активных ограничений (12.2).
- Множитель — одновременно сила и цена: $\mu^\ast=\partial f^\ast/\partial b$ локально (12.1, 12.2).